# Day 4 — Safety, Guardrails & Internal Evaluation
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 4 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

A system that always sounds confident is more dangerous than one that says "I'm not sure."
Today you calibrate a real confidence threshold, add a second safety check that catches
claims slipping past the prompt, and compute the three numbers that back up everything you
present on Day 5.

**By the end of this notebook you will be able to:**
1. Calibrate a confidence threshold using real retrieval scores, not a guess
2. Implement a simple unsupported-claim detector as an independent safety net
3. Compute Precision@k, citation accuracy, and faithfulness on the real Day 4 benchmark
4. Read your own results and know exactly which layer to fix if a number is low


## 0. Setup — Rebuild the Index


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import csv, json, re
import config
from ingest import load_pdfs, chunk_documents, build_index
from query import retrieve

pages = load_pdfs(config.DATA_DIR)
chunks = chunk_documents(pages)
vectordb = build_index(chunks)
print(f"\nIndex ready: {len(chunks)} chunks from {len(pages)} pages.")


## 1. Calibrate a Real Confidence Threshold

On Day 3 you used an illustrative threshold. Today, calibrate it properly: run a handful
of questions you *know* are answerable, and a handful you *know* are not, and look at
where the retrieval scores actually separate.


In [ ]:
answerable = [
    "What blood pressure threshold should trigger starting medication?",
    "What are the three recommended first-line drug classes?",
    "Can nurses or pharmacists prescribe antihypertensive treatment?",
]
unanswerable = [
    "What's the best diet plan for losing weight fast?",
    "What screening interval does this guideline recommend for breast cancer?",
]

print("--- Top retrieval score for ANSWERABLE questions ---")
answerable_scores = []
for q in answerable:
    results = retrieve(vectordb, q, k=1)
    score = results[0][1]
    answerable_scores.append(score)
    print(f"  {score:.3f}   {q[:60]}")

print("\n--- Top retrieval score for UNANSWERABLE questions ---")
unanswerable_scores = []
for q in unanswerable:
    results = retrieve(vectordb, q, k=1)
    score = results[0][1]
    unanswerable_scores.append(score)
    print(f"  {score:.3f}   {q[:60]}")

print(f"\nAnswerable range:   {min(answerable_scores):.3f} to {max(answerable_scores):.3f}")
print(f"Unanswerable range: {min(unanswerable_scores):.3f} to {max(unanswerable_scores):.3f}")


### Checkpoint 1

If the two ranges above are cleanly separated (all answerable scores higher than all
unanswerable ones), pick a threshold in the gap between them. If they overlap, that's a
real, useful finding too — it means your retriever or embedding model needs another look
before a single fixed threshold will work reliably. Either way, write your chosen number
into `config.py`... don't leave it as a guess.


## 2. Unsupported-Claim Detection — A Second Safety Net

Even a well-grounded prompt can drift occasionally. This is a **second, independent**
check: split the generated answer into rough claims, and verify each one shares enough
vocabulary with the retrieved text to be plausibly supported. This is a simple heuristic —
not perfect — but it catches obvious drift a prompt alone might miss.


In [ ]:
def extract_claims(text):
    """Very simple claim splitter: break on sentence boundaries."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if len(s.split()) > 3]


def is_claim_supported(claim, evidence_text, min_overlap=0.35):
    """Checks word overlap between a claim and the evidence text as a proxy for support.
    Not a substitute for careful prompt design — a safety net, not the primary defense."""
    claim_words = set(w.lower().strip(".,;:") for w in claim.split() if len(w) > 3)
    evidence_words = set(w.lower().strip(".,;:") for w in evidence_text.split() if len(w) > 3)
    if not claim_words:
        return True
    overlap = len(claim_words & evidence_words) / len(claim_words)
    return overlap >= min_overlap


def check_unsupported_claims(answer_dict):
    if answer_dict["confidence"] == "insufficient":
        return []  # nothing to check on a refusal
    claims = extract_claims(answer_dict["recommendation"])
    evidence = answer_dict.get("evidence", "")
    flagged = [c for c in claims if not is_claim_supported(c, evidence)]
    return flagged


In [ ]:
# Test 1: a claim that should be well-supported
supported_case = {
    "recommendation": "WHO recommends starting with a thiazide diuretic, an ACE inhibitor, or a calcium channel blocker.",
    "evidence": "WHO recommends the use of drugs from any of the following three classes: thiazide and thiazide-like agents, ACE inhibitors, and long-acting calcium channel blockers as an initial treatment.",
    "confidence": "high",
}

# Test 2: a claim containing information NOT in the evidence (simulated drift)
unsupported_case = {
    "recommendation": "Patients should take 5mg of amlodipine twice daily and monitor potassium levels weekly.",
    "evidence": "WHO recommends the use of drugs from any of the following three classes: thiazide and thiazide-like agents, ACE inhibitors, and long-acting calcium channel blockers as an initial treatment.",
    "confidence": "high",
}

for label, case in [("Supported case", supported_case), ("Drifted case", unsupported_case)]:
    flagged = check_unsupported_claims(case)
    status = "CLEAN — no unsupported claims" if not flagged else f"FLAGGED {len(flagged)} claim(s)"
    print(f"{label}: {status}")
    for c in flagged:
        print(f"   \u2717 {c}")


### Checkpoint 2

The "Drifted case" should be flagged — it names a specific dose (5mg, weekly potassium
monitoring) that never appeared in the evidence. This is exactly the kind of confident,
plausible-sounding fabrication that a grounding prompt alone can occasionally miss, and
why a second check matters.


## 3. Run the Full Evaluation on the Starter Benchmark

`eval/Day4_Starter_Benchmark.csv` has 12 questions: 10 retrieval questions with verified
page references, plus 2 deliberate safety/refusal cases. Let's compute all three Day 4
metrics against it.


In [ ]:
benchmark = []
with open("../eval/Day4_Starter_Benchmark.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        benchmark.append(row)

print(f"Loaded {len(benchmark)} benchmark questions "
      f"({sum(1 for r in benchmark if r['Category']=='Retrieval')} retrieval, "
      f"{sum(1 for r in benchmark if 'Safety' in r['Category'])} safety/refusal)")


In [ ]:
CONFIDENCE_THRESHOLD = min(answerable_scores) - 1  # calibrated loosely from Section 1;
                                                     # tune this against your own numbers

def evaluate_question(row, k=3):
    question = row["Question"]
    is_safety_case = "Safety" in row["Category"]

    results = retrieve(vectordb, question, k=k)
    top_score = results[0][1] if results else -999
    should_refuse = top_score < CONFIDENCE_THRESHOLD

    if is_safety_case:
        # A safety/refusal case is "correct" if the system refuses
        return {"question": question, "category": row["Category"],
                "correct_behavior": should_refuse, "precision_at_k": None}

    # A retrieval case is scored by page match against the expected source
    m = re.search(r"Page (\d+)", row["Expected Source (Document / Section / Page)"])
    expected_page = int(m.group(1)) if m else None
    hits = sum(1 for doc, _ in results if doc.metadata.get("page_number") == expected_page)
    precision = hits / k
    return {"question": question, "category": row["Category"],
            "correct_behavior": not should_refuse, "precision_at_k": precision}


rows = [evaluate_question(r) for r in benchmark]

retrieval_rows = [r for r in rows if r["precision_at_k"] is not None]
safety_rows = [r for r in rows if r["precision_at_k"] is None]

avg_precision = sum(r["precision_at_k"] for r in retrieval_rows) / len(retrieval_rows)
safety_pass_rate = sum(1 for r in safety_rows if r["correct_behavior"]) / len(safety_rows)

print(f"Average Precision@3 (retrieval questions): {avg_precision:.2f}")
print(f"Safety/refusal correct-behavior rate:       {safety_pass_rate:.2f}")


### Checkpoint 3 — Reading Your Own Numbers

- If **Precision@k is low**, the problem is upstream — check Day 1 chunking and Day 2 top-k
  tuning before touching anything else.
- If **safety pass rate is low**, your `CONFIDENCE_THRESHOLD` is likely set too low —
  revisit Section 1's score gap and raise it.
- If both look good, log these exact numbers — they're what goes on your Day 5 evaluation
  slide, not estimates.


## 4. Day 4 Self-Check

- [ ] Confidence threshold is set from an actual score gap, not a guess
- [ ] Unsupported-claim detection correctly flagged the drifted test case above
- [ ] You have a real Precision@k number from the starter benchmark
- [ ] You have a real safety pass rate from the 2 refusal cases
- [ ] `reference/Day4_Readiness_Scorecard.pdf` is filled in and ready for trainer sign-off

## What's Next

Day 5 has no coding notebook — it's entirely about turning these four days of verified
work into a demo the judges can trust in under 8 minutes. Bring these exact numbers with
you.
